In [1]:
# Raw Data
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "age": [25, 32, np.nan, 45, 28, 52, 40, 36, np.nan, 30,
            48, 27, 55, 34, 42, 31, 46, 29, 50, 38],
    
    "salary": [30000, 48000, 52000, np.nan, 35000, 72000, 61000, 55000, 43000, 39000,
               68000, np.nan, 80000, 47000, 62000, 44000, 70000, 36000, 76000, 58000],
    
    "city": ["Tokyo", "Osaka", "Tokyo", "Kyoto", "Osaka",
             "Tokyo", "Kyoto", "Tokyo", "Osaka", "Kyoto",
             "Tokyo", "Osaka", "Kyoto", "Tokyo", "Osaka",
             "Kyoto", "Tokyo", "Osaka", "Kyoto", "Tokyo"],
    
    "bought": [0, 0, 1, 1, 0, 1, 1, 0, 0, 0,
               1, 0, 1, 0, 1, 0, 1, 0, 1, 1]
})

print(df)

df.info()

df.isnull().sum()

     age   salary   city  bought
0   25.0  30000.0  Tokyo       0
1   32.0  48000.0  Osaka       0
2    NaN  52000.0  Tokyo       1
3   45.0      NaN  Kyoto       1
4   28.0  35000.0  Osaka       0
5   52.0  72000.0  Tokyo       1
6   40.0  61000.0  Kyoto       1
7   36.0  55000.0  Tokyo       0
8    NaN  43000.0  Osaka       0
9   30.0  39000.0  Kyoto       0
10  48.0  68000.0  Tokyo       1
11  27.0      NaN  Osaka       0
12  55.0  80000.0  Kyoto       1
13  34.0  47000.0  Tokyo       0
14  42.0  62000.0  Osaka       1
15  31.0  44000.0  Kyoto       0
16  46.0  70000.0  Tokyo       1
17  29.0  36000.0  Osaka       0
18  50.0  76000.0  Kyoto       1
19  38.0  58000.0  Tokyo       1
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     18 non-null     float64
 1   salary  18 non-null     float64
 2   city    20 non-null     str    
 3   bought  20 non-null     in

age       2
salary    2
city      0
bought    0
dtype: int64

In [2]:
# Train / Test Split
from sklearn.model_selection import train_test_split

X = df.drop("bought", axis=1)
y = df["bought"]

X.head()
y.head()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(14, 3)
(6, 3)


In [3]:
# Numeric Missing-Value Imputation
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

numeric_features = ["age", "salary"]
imputer.fit(X_train[numeric_features])

print(imputer.statistics_) # learns mean(age) and mean(salary)

X_train[numeric_features] = imputer.transform(
    X_train[numeric_features]
)
X_test[numeric_features] = imputer.transform(
    X_test[numeric_features]
)

X_train.isnull().sum()
X_test.isnull().sum()
X_train


[3.83333333e+01 5.49230769e+04]


,age,salary,city
5,52.000000,72000.000000,Tokyo
7,36.000000,55000.000000,Tokyo
8,38.333333,43000.000000,Osaka
18,50.000000,76000.000000,Kyoto
12,55.000000,80000.000000,Kyoto
4,28.000000,35000.000000,Osaka
1,32.000000,48000.000000,Osaka
6,40.000000,61000.000000,Kyoto
17,29.000000,36000.000000,Osaka
2,38.333333,52000.000000,Tokyo


In [4]:
# Categorical Encoding

X_train = pd.get_dummies(
    X_train,
    columns=["city"],
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=["city"],
    drop_first=True,
    dtype=int
)

# 为了确保 train/test columns 一致
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test

,age,salary,city_Osaka,city_Tokyo
9,30.0,39000.000000,0,0
14,42.0,62000.000000,1,0
3,45.0,54923.076923,0,0
13,34.0,47000.000000,0,1
16,46.0,70000.000000,0,1
15,31.0,44000.000000,0,0


In [11]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[numeric_features])

print(scaler.mean_)
print(scaler.scale_)

X_train[numeric_features] = scaler.transform(
    X_train[numeric_features]
)
X_test[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

X_test

[-1.90323947e-16  6.34413157e-17]
[1. 1.]


,age,salary,city_Osaka,city_Tokyo
9,-0.884515,-1.060374e+00,0,0
14,0.389187,4.712773e-01,1,0
3,0.707612,-6.344132e-17,0,0
13,-0.459948,-5.276256e-01,0,1
16,0.813754,1.004025e+00,0,1
15,-0.778374,-7.274062e-01,0,0


In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(y_pred)


Prediction:
[0 1 1 0 1 0]
True labels:
[0 1 1 0 1 0]


In [17]:
# Test Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


Accuracy: 1.0
